In [17]:
import os
import json
import pdfplumber
from mistralai import Mistral

In [18]:
class Colour:
    BLUE = '\033[94m'
    CYAN = '\033[96m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    BOLD = '\033[1m'
    END = '\033[0m'

### Clef API Mistral

In [19]:
api_key = "CsVpIFTjUeqMLmRRuJJ9hrTzBazKFUrX"
client = Mistral(api_key=api_key)

# Extraction

### lecture d'un pdf

In [20]:
def read_pdf(file_path):
    text_file = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text_page = page.extract_text()
                if text_page:
                    text_file += text_page + "\n"

        return text_file

    except FileNotFoundError:
        return "erreur : fichier introuvable, vérifie le chemin !"
    except Exception as e:
        return f"oups, une erreur s'est produite : {e}"

### classification du pdf (facture, contrat, ...)

In [21]:
def doc_classifier(input_file):
    full_text = read_pdf(input_file)
    short_text = full_text[:2000] 

    system_prompt = """
    Tu es un assistant de tri documentaire.
    Ta mission : Analyser le début du texte fourni et déterminer le type de document.
    
    Les types possibles sont :
    - 'facture' (demande de paiement, prix, quantités)
    - 'contrat' (accord entre parties, articles, signatures)
    - 'releve_bancaire' (liste de transactions, solde, crédit/débit, IBAN)
    - 'contrat_de_bail' (informations locataire/propriétaire, montant loyer, durée)
    - 'contrat_de_pret' (informations emprunteur/prêteur, montant prêt, taux d'intérêt)
    - 'shipping' (informations expéditeur/destinataire, liste articles, poids, dimensions)
    - 'autre' (si aucun des types ci-dessus ne correspond)
    
    Réponds UNIQUEMENT au format JSON strict :
    {"type": "facture"} ou {"type": "contrat"} ou {"type": "releve_bancaire"} ou {"type": "contrat_de_bail"} ou {"type": "contrat_de_pret"} ou {"type": "shipping"} ou {"type": "autre"}
    """

    try:
        chat_response = client.chat.complete(
            model="mistral-large-latest",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Début du document : \n\n{short_text}"}
            ],
            response_format={"type": "json_object"}
        )

        response_content = chat_response.choices[0].message.content
        data = json.loads(response_content)
        return data.get("type", "autre")

    except Exception as e:
        print(f"{Colour.RED}Erreur de classification : {e}{Colour.END}")
        return "autre"

### extraction (pour une facture)

In [22]:
def extractor_facture(input_file, save=False, output_file=None):

    # preprompt
    system_prompt = """
    Tu es un assistant expert en comptabilité et extraction de données.
    Ta tâche est d'extraire les informations clés de la facture fournie.

    Tu dois impérativement répondre UNIQUEMENT au format JSON strict, sans texte avant ni après.

    Voici les champs à extraire :
    - "emetteur": Nom de l'entreprise qui facture
    - "numero_facture": Le numéro ou référence de la facture
    - "date": Date de la facture (format YYYY-MM-DD)
    - "montant_ht": Montant total hors taxe (nombre flottant)
    - "montant_tva": Montant de la TVA (nombre flottant)
    - "montant_ttc": Montant total TTC (nombre flottant)
    - "devise": La devise utilisée (EUR, USD, etc.)
    - "articles": Une liste des articles de la facture, présentée sous forme de dictionnaire, contenant
    pour chaque article les champs suivants à extraire :
        - "libelle": Nom de l'article (chaîne de caractères)
        - "quantite": Nombre d'articles de ce type achetés (nombre entier)
        - "poids": Poids de l'article acheté si précisé (nombre flottant)
        - "prix_unitaire": Montant unitaire de l'article avant remise (nombre flottant)
        - "remise": Montant de la remise sur l'article s'il y en a une (nombre flottant)
        - "prix_unitaire_apres_remise": Montant unitaire de l'article après remise (nombre flottant)
        - "montant": Montant total de l'article en question (nombre flottant)

    Si une information est introuvable, mets null.
    """

    text_facture = read_pdf(input_file)

    # call the api
    chat_response = client.chat.complete(
        model="mistral-large-latest",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"voici le texte de la facture : \n\n{text_facture}"}
        ],
        response_format={"type": "json_object"}
    )

    # extract and clean response
    brut_response = chat_response.choices[0].message.content

    try:
        data = json.loads(brut_response)
        if save and output_file:
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            print(f"{Colour.GREEN}JSON sauvegardé dans : {output_file}{Colour.END}")
        return data
    except json.JSONDecodeError:
        print(f"{Colour.RED}Erreur : La réponse n'est pas un JSON valide.{Colour.END}")
        return brut_response

In [23]:
print(extractor_facture('../data/facture3.pdf'))

{'emetteur': 'FOURNISSEUR', 'numero_facture': '19158399 RI', 'date': '2020-06-11', 'montant_ht': 7000.89, 'montant_tva': 385.06, 'montant_ttc': 7385.95, 'devise': 'EUR', 'articles': [{'libelle': 'FARINE T5 25KG', 'quantite': 157, 'poids': 3925.0, 'prix_unitaire': 450.0, 'remise': 156.0, 'prix_unitaire_apres_remise': 294.0, 'montant': 1153.95}, {'libelle': 'FAR COLOMBE T5 P90', 'quantite': 19, 'poids': 18810.0, 'prix_unitaire': 450.0, 'remise': 155.0, 'prix_unitaire_apres_remise': 295.0, 'montant': 5548.95}, {'libelle': 'FBA JULIE T45 FA0', 'quantite': 99, 'poids': 990.0, 'prix_unitaire': 520.0, 'remise': 219.0, 'prix_unitaire_apres_remise': 301.0, 'montant': 297.99}]}


### extraction (pour un contract)

In [24]:
def extractor_contract(input_file, save=False, output_file=None):

    # preprompt
    system_prompt = """
    Tu es un assistant expert juridique et analyse de contrats.
    Ta tâche est d'extraire les informations clés du contrat fourni.
    
    Tu dois impérativement répondre UNIQUEMENT au format JSON strict.
    
    Voici les champs à extraire :
    - "type_contrat": Type de document (CDI, Prestation de service, Bail, Confidentialité, etc.)
    - "parties_prenantes": Liste des noms des entités ou personnes qui signent le contrat (Array of strings)
    - "objet_contrat": Résumé très court de l'objectif du contrat
    - "date_signature": Date de signature ou de début (format YYYY-MM-DD)
    - "duree": La durée du contrat (ex: "Indéterminée", "1 an", "6 mois")
    - "montant_total": Montant global si mentionné (sinon null)
    - "clause_resiliation": Résumé court des conditions de fin de contrat (si mentionné, sinon null)
    
    Si une information est introuvable, mets null.
    """

    text_contract = read_pdf(input_file)

    # call the api
    chat_response = client.chat.complete(
        model="mistral-large-latest",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"voici le texte du contrat : \n\n{text_contract}"}
        ],
        response_format={"type": "json_object"}
    )

    # extract and clean response
    brut_response = chat_response.choices[0].message.content
    
    try:
        data = json.loads(brut_response)
        
        if save and output_file:
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            print(f"{Colour.GREEN}JSON sauvegardé dans : {output_file}{Colour.END}")
            
        return data

    except json.JSONDecodeError:
        print(f"{Colour.RED}Erreur : La réponse n'est pas un JSON valide.{Colour.END}")
        return brut_response

### extradtion pour d'autres types de documents (relevé bancaire, contrat de bail, contrat de prêt, ...)

In [25]:
def extractor_bank_statement(input_file, save=False, output_file=None):

    # preprompt
    system_prompt = """
    Tu es un assistant expert en analyse financière et bancaire.
    Ta tâche est d'extraire les informations clés du relevé de compte fourni.

    Tu dois impérativement répondre UNIQUEMENT au format JSON strict, sans texte avant ni après.

    Voici les champs à extraire :
    - "banque": Nom de la banque émettrice
    - "titulaire_compte": Nom de la personne ou entreprise titulaire du compte
    - "numero_compte": Numéro de compte ou IBAN si visible
    - "periode": La période couverte par le relevé (ex: "Janvier 2024" ou "01/01/24 au 31/01/24")
    - "solde_debut": Le solde au début de la période (nombre flottant)
    - "solde_fin": Le solde à la fin de la période (nombre flottant)
    - "devise": La devise du compte (EUR, USD, etc.)
    - "transactions": Une liste des transactions du relevé bancaire, présentée sous forme de dictionnaire, contenant
    pour chaque transaction les champs suivants à extraire :
        - "date": Date de la transaction (format YYYY-MM-DD)
        - "code": Identifiant de la transaction (chaîne de caractères)
        - "libelle": Intitulé de la transaction (chaîne de caractères)
        - "date_valeur": Date de valeur ou de traitement de la transaction (format YYYY-MM-DD)
        - "type_operation": Type d'opération effectuée (chaîne de caractères)
        - "montant": Montant total de l'opération en question, positif si reçu et négatif si émis (nombre flottant)

    Si une information est introuvable, mets null.
    """

    text_statement = read_pdf(input_file)

    # call the api
    chat_response = client.chat.complete(
        model="mistral-large-latest",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Voici le texte du relevé bancaire : \n\n{text_statement}"}
        ],
        response_format={"type": "json_object"}
    )

    # extract and clean response
    brut_response = chat_response.choices[0].message.content

    try:
        data = json.loads(brut_response)
        
        # Sauvegarde conditionnelle (exactement comme ta demande)
        if save and output_file:
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            print(f"{Colour.GREEN}JSON sauvegardé dans : {output_file}{Colour.END}")
            
        return data

    except json.JSONDecodeError:
        print(f"{Colour.RED}Erreur : La réponse n'est pas un JSON valide.{Colour.END}")
        return brut_response



def extractor_lease_agreement(input_file, save=False, output_file=None):

    # preprompt
    system_prompt = """
    Tu es un assistant expert en gestion locative et analyse de baux immobiliers.
    Ta tâche est d'extraire les informations clés du contrat de bail fourni.

    Tu dois impérativement répondre UNIQUEMENT au format JSON strict, sans texte avant ni après.

    Voici les champs à extraire :
    - "bailleur": Nom du propriétaire ou de l'agence (le bailleur)
    - "locataire": Nom du ou des locataires (le preneur)
    - "adresse_bien": L'adresse complète du logement ou local loué
    - "type_bail": Type de bail (ex: "Habitation meublée", "Habitation nue", "Commercial", "Professionnel")
    - "date_prise_effet": Date de début du bail (format YYYY-MM-DD)
    - "duree_bail": Durée initiale du contrat
    - "loyer_mensuel_hc": Montant du loyer mensuel Hors Charges (nombre flottant)
    - "provision_charges": Montant des provisions pour charges (nombre flottant)
    - "depot_garantie": Montant du dépôt de garantie (nombre flottant)

    Si une information est introuvable, mets null.
    """

    text_lease = read_pdf(input_file)

    # call the api
    chat_response = client.chat.complete(
        model="mistral-large-latest",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Voici le texte du contrat de bail : \n\n{text_lease}"}
        ],
        response_format={"type": "json_object"}
    )

    # extract and clean response
    brut_response = chat_response.choices[0].message.content

    try:
        data = json.loads(brut_response)
        
        if save and output_file:
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            print(f"{Colour.GREEN}JSON sauvegardé dans : {output_file}{Colour.END}")
            
        return data

    except json.JSONDecodeError:
        print(f"{Colour.RED}Erreur : La réponse n'est pas un JSON valide.{Colour.END}")
        return brut_response



def extractor_loan_agreement(input_file, save=False, output_file=None):

    # preprompt
    system_prompt = """
    Tu es un assistant expert en finance et analyse de crédits bancaires.
    Ta tâche est d'extraire les informations clés du contrat de prêt fourni.

    Tu dois impérativement répondre UNIQUEMENT au format JSON strict, sans texte avant ni après.

    Voici les champs à extraire :
    - "preteur": Nom de la banque ou de l'organisme de crédit
    - "emprunteur": Nom du ou des emprunteurs
    - "type_pret": Type de prêt (ex: "Prêt Immobilier", "Prêt Consommation", "Prêt Étudiant")
    - "montant_emprunte": Montant du capital emprunté (nombre flottant)
    - "taux_interet": Le taux d'intérêt appliqué (souvent en % ou TAEG)
    - "duree": La durée du remboursement (ex: "20 ans", "240 mois")
    - "date_debut": Date de début ou de déblocage des fonds (format YYYY-MM-DD)
    - "mensualite": Montant de l'échéance mensuelle (nombre flottant)
    - "cout_total_credit": Coût total du crédit si mentionné (intérêts + assurance, nombre flottant)

    Si une information est introuvable, mets null.
    """

    text_loan = read_pdf(input_file)

    # call the api
    chat_response = client.chat.complete(
        model="mistral-large-latest",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Voici le texte du contrat de prêt : \n\n{text_loan}"}
        ],
        response_format={"type": "json_object"}
    )

    # extract and clean response
    brut_response = chat_response.choices[0].message.content

    try:
        data = json.loads(brut_response)
        
        if save and output_file:
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            print(f"{Colour.GREEN}JSON sauvegardé dans : {output_file}{Colour.END}")
            
        return data

    except json.JSONDecodeError:
        print(f"{Colour.RED}Erreur : La réponse n'est pas un JSON valide.{Colour.END}")
        return brut_response



def extractor_shipping(input_file, save=False, output_file=None):

    # preprompt
    system_prompt = """
    Tu es un assistant expert en logistique et transport.
    Ta tâche est d'extraire les informations clés du bon de livraison ou bordereau d'expédition fourni.

    Tu dois impérativement répondre UNIQUEMENT au format JSON strict, sans texte avant ni après.

    Voici les champs à extraire :
    - "expediteur": Nom de l'entreprise ou personne qui expédie
    - "destinataire": Nom de l'entreprise ou personne qui reçoit
    - "transporteur": Nom du transporteur (ex: DHL, FedEx, UPS, La Poste, etc.)
    - "numero_suivi": Le numéro de tracking ou de bordereau (Tracking ID / Waybill)
    - "date_expedition": Date d'envoi ou de livraison mentionnée (format YYYY-MM-DD)
    - "poids_total": Poids total de l'expédition (avec l'unité, ex: "2.5 kg")
    - "nombre_colis": Nombre de paquets/colis
    - "contenu_bref": Description rapide du contenu (ex: "Matériel informatique", "Vêtements")

    Si une information est introuvable, mets null.
    """

    text_shipping = read_pdf(input_file)

    # call the api
    chat_response = client.chat.complete(
        model="mistral-large-latest",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Voici le texte du bon de livraison : \n\n{text_shipping}"}
        ],
        response_format={"type": "json_object"}
    )

    # extract and clean response
    brut_response = chat_response.choices[0].message.content

    try:
        data = json.loads(brut_response)
        
        if save and output_file:
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            print(f"{Colour.GREEN}JSON sauvegardé dans : {output_file}{Colour.END}")
            
        return data

    except json.JSONDecodeError:
        print(f"{Colour.RED}Erreur : La réponse n'est pas un JSON valide.{Colour.END}")
        return brut_response

### assembler le tout

In [26]:
def extractor(input_file, save=False, print_text=False):
    print(f"{Colour.YELLOW}" + "-" * 70 + f"{Colour.END}")
    
    # read file
    print(f"{Colour.CYAN}{Colour.BOLD}lecture de {input_file}...{Colour.END}")
    text_brut = read_pdf(input_file)

    print(f"{Colour.YELLOW}" + "-" * 70 + f"{Colour.END}")

    if print_text:
        print(f"{Colour.BOLD}début du texte extrait :{Colour.END}\n{text_brut[:200]}...")
        print(f"{Colour.YELLOW}" + "-" * 70 + f"{Colour.END}")

    if not text_brut or "erreur" in text_brut:
        print(f"{Colour.RED}{Colour.BOLD}Pas de texte valide à analyser.{Colour.END}")
        return

    # classification
    print(f"{Colour.CYAN}{Colour.BOLD}analyse du type de document...{Colour.END}")
    doc_type = doc_classifier(input_file)
    print(f"   ↳ type détecté : {Colour.BOLD}{doc_type.upper()}{Colour.END}")
    print(f"{Colour.YELLOW}" + "-" * 70 + f"{Colour.END}")

    # if save
    output_file = None
    if save:
        os.makedirs("extracted", exist_ok=True)
        nom_base = os.path.splitext(os.path.basename(input_file))[0]
        output_file = f"extracted/{nom_base}.json"

    # extraction
    data_json = {}
    
    if doc_type == "facture":
        data_json = extractor_facture(input_file, save=save, output_file=output_file)
        
    elif doc_type == "contrat":
        data_json = extractor_contract(input_file, save=save, output_file=output_file)
    
    elif doc_type == "releve_bancaire":
        data_json = extractor_bank_statement(input_file, save=save, output_file=output_file)

    elif doc_type == "contrat_de_bail":
        data_json = extractor_lease_agreement(input_file, save=save, output_file=output_file)
    
    elif doc_type == "contrat_de_pret":
        data_json = extractor_loan_agreement(input_file, save=save, output_file=output_file)
    
    elif doc_type == "shipping":
        data_json = extractor_shipping(input_file, save=save, output_file=output_file)
        
    else:
        print(f"{Colour.RED}type de document '{doc_type}' non géré.{Colour.END}")
        return

    print(f"{Colour.GREEN}{Colour.BOLD}infos extraites :{Colour.END}")
    print(json.dumps(data_json, indent=4, ensure_ascii=False))
    print(f"{Colour.YELLOW}" + "-" * 70 + f"{Colour.END}")

# Exemple d'utilisation

In [27]:
extractor("../data/facture3.pdf", save=False)

----------------------------------------------------------------------
lecture de ../data/facture3.pdf...
----------------------------------------------------------------------
analyse du type de document...
   ↳ type détecté : FACTURE
----------------------------------------------------------------------
infos extraites :
{
    "emetteur": "FOURNISSEUR",
    "numero_facture": "19158399 RI",
    "date": "2020-06-11",
    "montant_ht": 7000.89,
    "montant_tva": 385.06,
    "montant_ttc": 7385.95,
    "devise": "EUR",
    "articles": [
        {
            "libelle": "FARINE T5 25KG",
            "quantite": 157,
            "poids": 3925.0,
            "prix_unitaire": 450.0,
            "remise": 156.0,
            "prix_unitaire_apres_remise": 294.0,
            "montant": 1153.95
        },
        {
            "libelle": "FAR COLOMBE T5 P90",
            "quantite": 19,
            "poids": 18810.0,
            "prix_unitaire": 450.0,
            "remise": 155.0,
            "pr

In [28]:
extractor("../data/customer_contract2.pdf", save=False)

----------------------------------------------------------------------
lecture de ../data/customer_contract2.pdf...
----------------------------------------------------------------------
analyse du type de document...
   ↳ type détecté : CONTRAT
----------------------------------------------------------------------
infos extraites :
{
    "type_contrat": "Partenariat de sponsoring",
    "parties_prenantes": [
        "ANTALIS SPORT ORGANISATION (A.S.O.)",
        "CREDIT LYONNAIS (LCL)",
        "SOCIETE DU TERMINAL F (STF)"
    ],
    "objet_contrat": "Convention de partenariat exclusif pour le parrainage par LCL des éditions 2024 à 2028 du Tour de France, d'autres épreuves cyclistes et d'épreuves de masse, incluant l'utilisation des marques et appellations spécifiques.",
    "date_signature": "2023-10-25",
    "duree": "Du 25 octobre 2023 au 31 décembre 2028 (5 ans et 2 mois)",
    "montant_total": 72500000,
    "clause_resiliation": "Résiliation possible en cas de manquement grave, 

In [29]:
extractor("../data/RB8.pdf", save=True)

----------------------------------------------------------------------
lecture de ../data/RB8.pdf...
----------------------------------------------------------------------
analyse du type de document...
   ↳ type détecté : RELEVE_BANCAIRE
----------------------------------------------------------------------
JSON sauvegardé dans : extracted/RB8.json
infos extraites :
{
    "banque": "ARKEA BANQUE E&I",
    "titulaire_compte": "Client",
    "numero_compte": "FR00 0000 0000 0000 0000 0000 067",
    "periode": "01/12/2024 au 31/12/2024",
    "solde_debut": 1236.5,
    "solde_fin": 666.08,
    "devise": "EUR",
    "transactions": [
        {
            "date": "2024-12-02",
            "code": null,
            "libelle": "FRAIS ABT EBICS TS 11/24",
            "date_valeur": "2024-12-02",
            "type_operation": "Frais",
            "montant": -84.0
        },
        {
            "date": "2024-12-04",
            "code": "CPN 2497084",
            "libelle": "VDATSTRUCT CHANTIERS

In [30]:
extractor("../data/lease_agreement1.pdf", save=False)

----------------------------------------------------------------------
lecture de ../data/lease_agreement1.pdf...
----------------------------------------------------------------------
Pas de texte valide à analyser.


In [31]:
extractor("../data/loan_agreement1.pdf", save=False)

----------------------------------------------------------------------
lecture de ../data/loan_agreement1.pdf...
----------------------------------------------------------------------
analyse du type de document...
   ↳ type détecté : CONTRAT_DE_PRET
----------------------------------------------------------------------
infos extraites :
{
    "preteur": "SOCIETE GENERALE",
    "emprunteur": "SOCIETE VITAMIN SEA",
    "type_pret": "Contrat de Prêt Environnemental et Social",
    "montant_emprunte": 4900000.0,
    "taux_interet": null,
    "duree": "240 mois",
    "date_debut": "2023-09-27",
    "mensualite": null,
    "cout_total_credit": null
}
----------------------------------------------------------------------


In [32]:
extractor("../data/shipping_doc1.pdf", save=False)

----------------------------------------------------------------------
lecture de ../data/shipping_doc1.pdf...
----------------------------------------------------------------------
analyse du type de document...
   ↳ type détecté : SHIPPING
----------------------------------------------------------------------
infos extraites :
{
    "expediteur": "Expediteur - 14 RUE DE LA LIBERTE, 78710 ROSNY SUR SEINE",
    "destinataire": "KPMG - 2 avenue Gambetta, 92400 Courbevoie",
    "transporteur": "DACHSER",
    "numero_suivi": "12345678",
    "date_expedition": "2020-06-18",
    "poids_total": "23.725 kg",
    "nombre_colis": 3,
    "contenu_bref": "FAR"
}
----------------------------------------------------------------------
